# 🌊 Huấn Luyện RT-DETR Nhận Diện Ngập Lụt trên Kaggle (GPU T4 x2 / P100)

Notebook này hướng dẫn quy trình huấn luyện mô hình **RT-DETR (Ultralytics)** trên nền tảng **Kaggle** hoàn toàn miễn phí.

### ⚠️ Lưu ý trước khi chạy:
1. Bật GPU trong Kaggle Notebook: **Settings** -> **Accelerator** -> Chọn **GPU T4 x2** hoặc **GPU P100**.
2. Bật mạng: **Settings** -> **Internet** -> Chọn **Internet on**.

## Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q ultralytics onnx onnxslim

## Bước 2: Kiểm tra GPU

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

## Bước 3: Tạo file cấu hình data.yaml trên Kaggle
> **Lưu ý**: Nếu bạn upload dataset lên Kaggle Dataset, đường dẫn sẽ nằm tại `/kaggle/input/<ten-dataset-cua-ban>/`.
> Đoạn code dưới đây tạo file `data_kaggle.yaml` trỏ đúng vào thư mục dataset đó.

In [ ]:
import os

# Đổi tên 'flood-dataset' thành tên dataset bạn đã tải lên Kaggle (nếu có)
DATASET_DIR = "/kaggle/input/flood-dataset"

yaml_content = f"""
path: {DATASET_DIR}
train: images/train
val: images/val

names:
  0: duong_ngap
  1: xe_ngap
  2: nguoi_loi_nuoc
"""

with open("/kaggle/working/data_kaggle.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("✅ Đã tạo file cấu hình: /kaggle/working/data_kaggle.yaml")

## Bước 4: Bắt đầu Huấn luyện (Training)

In [ ]:
from ultralytics import RTDETR

# Khởi tạo pre-trained RT-DETR-L
model = RTDETR("rtdetr-l.pt")

# Huấn luyện mô hình
results = model.train(
    data="/kaggle/working/data_kaggle.yaml",
    epochs=100,
    imgsz=640,
    batch=16,          # Kaggle T4 x2 hoặc P100 có 16GB VRAM, chạy rất tốt ở batch=16
    device=0,
    workers=4,
    project="/kaggle/working/runs_flood",
    name="rtdetr_kaggle_exp",
    exist_ok=True,
    save=True
)

print("✅ Huấn luyện hoàn tất! Trọng số lưu tại: /kaggle/working/runs_flood/rtdetr_kaggle_exp/weights/best.pt")

## Bước 5: Export sang định dạng ONNX
Sau khi huấn luyện xong, chạy cell này để xuất sang file `rt_detr_flood.onnx` chuẩn bị tải về máy.

In [ ]:
import os
import shutil
from ultralytics import RTDETR

best_pt = "/kaggle/working/runs_flood/rtdetr_kaggle_exp/weights/best.pt"
model = RTDETR(best_pt)

# Export ONNX
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    opset=16,
    batch=1
)

# Đổi tên và đưa ra thư mục /kaggle/working/ để dễ tải về
final_onnx = "/kaggle/working/rt_detr_flood.onnx"
shutil.copy2(onnx_path, final_onnx)

print(f"🎉 Hoàn tất! File ONNX đã sẵn sàng tại: {final_onnx}")
print("👉 Bạn chỉ cần vào tab Output bên phải màn hình Kaggle và click Download file 'rt_detr_flood.onnx'.")